# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ziadhamouda370-beep/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.FlyRank reports that growing pages were younger on average than declining pages: 185 days versus 228 days, while average word count was almost identical at about 1.5K words. The outcome label comes from grouping pages by whether traffic was gaining or losing. The validation supports this as an observed portfolio pattern, but it does not prove that age causes decline because the study is observational and the paper notes that content age can confound comparisons.
 FlyRank reports that pages in the 31–90 day freshness window had a 5.43:1 growth-to-decline ratio. A separate comparison of 365+ pages found higher health and impressions for recently refreshed pages, but the paper explicitly treats these as observational comparisons rather than causal proof. The validation design supports a directional refresh hypothesis, but a stronger causal claim would require a controlled or carefully matched experiment.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*I will evaluate my Random Forest using a client-holdout split, with approximately 20% of clients kept completely outside the training data. I will compare the model with the earlier evaluation using the same target and metric. The grouped split is more honest because content from the same client cannot appear in both training and test sets.

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score

# Load data
url = "https://raw.githubusercontent.com/ziadhamouda370-beep/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# Target
df["is_declining"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# Features used for the model
candidate_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate"
]

feature_cols = [
    c for c in candidate_features
    if c in df.columns
]

# Honest client-level split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train = train_df["is_declining"]
y_test = test_df["is_declining"]

# Missing values
medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(medians).fillna(0)
X_test = X_test.fillna(medians).fillna(0)

# Train Random Forest
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

# Predictions
model_probability = model.predict_proba(X_test)[:, 1]

test_predictions = (
    model_probability >= 0.5
).astype(int)

# Metrics
honest_recall = recall_score(
    y_test,
    test_predictions,
    zero_division=0
)

honest_precision = precision_score(
    y_test,
    test_predictions,
    zero_division=0
)

# Ranking
model_results = test_df[
    ["content_id", "client_id", "is_declining"]
].copy()

model_results["model_score"] = model_probability

model_results = model_results.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

# Precision@50
precision_at_50 = (
    model_results.head(50)["is_declining"].mean()
)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

print("Client overlap:",
      len(
          set(train_df["client_id"]) &
          set(test_df["client_id"])
      ))

print("\nHonest client-holdout validation")
print("Recall:", round(honest_recall, 3))
print("Precision:", round(honest_precision, 3))
print("Precision@50:", round(precision_at_50, 3))

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0

Honest client-holdout validation
Recall: 0.928
Precision: 0.894
Precision@50: 1.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*
I checked the final feature set for leakage. The target field trend_direction and derived target fields are excluded from the model features. Product or decision flags are also excluded because they can encode information about the outcome. The client identifier is used only for grouping the train/test split, not as a feature. No future-window outcome is intentionally used as a model feature.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Final feature-set leakage audit

forbidden_fields = [
    "trend_direction",
    "trend_pct",
    "health_score",
    "needs_ctr_fix",
    "is_quick_win",
    "client_id"
]

leaked_fields = [
    col for col in forbidden_fields
    if col in feature_cols
]

print("Number of model features:", len(feature_cols))
print("Potential forbidden fields found:", leaked_fields)

print("\nFeature list:")
print(feature_cols)

assert "trend_direction" not in feature_cols
assert "trend_pct" not in feature_cols
assert "client_id" not in feature_cols

print("\nLeakage checks passed.")

Number of model features: 26
Potential forbidden fields found: []

Feature list:
['search_volume', 'competition', 'cpc', 'word_count', 'content_age_days', 'days_since_last_update', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']

Leakage checks passed.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*Safer claim: The model measured patterns associated with the starter dataset's declining-content label and produced a directional ranking for content-review prioritization. The result is decision-support evidence and does not prove causality or predict Google's ranking algorithm.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
original_claim = (
    "The model predicts which content will decline in Google search."
)

safe_claim = (
    "The model measured patterns associated with the starter dataset's "
    "declining-content label and produced a directional ranking for "
    "content-review prioritization. The result is decision-support evidence "
    "and does not prove causality or predict Google's ranking algorithm."
)

print("Original claim:")
print(original_claim)

print("\nSafer claim:")
print(safe_claim)

Original claim:
The model predicts which content will decline in Google search.

Safer claim:
The model measured patterns associated with the starter dataset's declining-content label and produced a directional ranking for content-review prioritization. The result is decision-support evidence and does not prove causality or predict Google's ranking algorithm.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.